# Data loading

In [ ]:
import json
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer


The trial release includes: **one ALD/E research paper**, and its associated **annotated scientific figures**, with a focus on quantitative plots.

### 🗂️ Directory Structure

```text
icdar2026-competition-data
├── trial
│   ├── main-category
│   │   ├── sub-category
│   │   │   ├── paper #
│   │   │   │   ├── images
│   │   │   │   │   ├── filename.jpg            # (JPEG) actual figure image
│   │   │   │   │   ├── filename.json           # (JSON) figure annotations
│   │   │   │   │   └── ...
│   │   │   │   ├── Author et al.pdf                # (PDF) actual PDF document
│   │   │   │   ├── content.json                    # (JSON) structured content
│   │   │   │   └── ...
│   │   │   └── ...
```

In [ ]:
!git clone https://github.com/sciknoworg/ALD-E-ImageMiner/ ../ALD-E-ImageMiner

In [ ]:
MODEL_ID = "unsloth/Qwen3.5-0.8B"

BASE_DIR = Path.cwd().parent
CATEGORY = "train"

COMPETITION_DATA_DIR = BASE_DIR / "ALD-E-ImageMiner" / "icdar2026-competition-data"
CASE_DIR = COMPETITION_DATA_DIR / CATEGORY

assert CASE_DIR.is_dir(), "Dataset not found"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


### 📝 Annotations

Following is the general top-level schema of each JSON file. Each "dict" contains alphabetical keys-values pairs to represent each sub-figure task-specific data as shown below, where

- **sample_id**: Unique sample id, represents actual path to the figure
- **classification**: Data for the classification task 
- **summarization**: Data for the Summarization task
- **data_extraction**: Data for the Data extraction task
- **vqa**: Data for the VQA task
- **bbox**: Bounding box coordinates to extract each sub-figure

```json
{
    "sample_id": str,
    "classification": {
        "a": str,
        "b": str,
        ...
    },
    "summarization": {
        "a": str,
        "b": str,
        ...
    },
    "data_extraction": {
        "a": str,
        "b": str,
        ...
    },
    "vqa": {
        "a": {
            "question_type": str,
            "questions": str,
            "answer_type": str,
            "answer": str
        },
        "b": list,
        ...
    },
    "bbox": {
        "a": {
            "x": int,
            "y": int,
            "width": int,
            "height": int
        },
        "b": {
            "x": int,
            "y": int,
            "width": int,
            "height": int
        },
        ...
    },
}
```

In [ ]:
!ls -R {CASE_DIR}

In [ ]:
!cat {CASE_DIR}/atomic-layer-etching/experimental-usecase/16/images/fig1.json

In [ ]:
def analyze_dataset(case_dir: Path, tokenizer, max_samples=None):
    stats = []
    json_files = list(case_dir.rglob("*.json"))

    if max_samples:
        json_files = json_files[:max_samples]

    pbar = tqdm(json_files, desc="Analyzing Data")

    for json_file in pbar:
        fullpath = str(json_file)
        if "images" not in fullpath or ".vscode" in fullpath:
            continue

        pbar.set_description(f"Processing {json_file.name}")

        with open(json_file, "r") as f:
            data = json.load(f)

        bboxes = data.get("bbox", {})

        for sub_key, q_list in data.get("vqa", {}).items():
            if sub_key not in bboxes:
                continue

            # Get coordinates to calculate crop size
            box = bboxes[sub_key]
            width = box["width"]
            height = box["height"]

            for q_obj in q_list:
                gt_response = q_obj.get("answer", "")

                answer_tokens = len(
                    tokenizer.encode(gt_response, add_special_tokens=False)
                )

                stats.append(
                    {
                        "file": json_file.name,
                        "subfigure": sub_key,
                        "answer_tokens": answer_tokens,
                        "image_width": width,
                        "image_height": height,
                        "image_pixels": width * height,
                    }
                )

    return pd.DataFrame(stats)

In [ ]:
df_stats = analyze_dataset(CASE_DIR, tokenizer)

### 📊 Text Token Statistics

Let's look at the distribution of text tokens. This tells us how much context length is consumed purely by the text prompt and the desired output.

> 1. To set `MAX_NEW_TOKENS`: Look at the 99th percentile of `answer_tokens`.
> 2. To set `max_length` in `SFTConfig`: Add the 99th percentile of `answer_tokens` + `prompt_tokens` + `visual_tokens` to the estimated visual tokens. 
> 
> *Qwen typically encodes images dynamically. If your average image is ~1000x1000, it can easily take up 1000+ tokens on its own. A safe `max_length` is usually between 2048 and 4096 depending on your hardware limits.*

In [ ]:
display(df_stats["answer_tokens"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

In [ ]:
df_stats["answer_tokens"].hist(bins=50, color="skyblue", edgecolor="black")
plt.show()

### 🖼️ Image Dimension Statistics

Qwen-VL processes images by splitting them into patches. Larger images consume exponentially more sequence length. If your subfigure crops are massive, you may need a much higher `max_length` or you might need to resize them before training.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].scatter(
    df_stats["image_width"], df_stats["image_height"], alpha=0.5, color="green"
)
axes[0].set_title("Image Crop Dimensions (Width vs Height)")
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Height (px)")

df_stats["image_pixels"].hist(bins=50, ax=axes[1], color="purple", edgecolor="black")
axes[1].set_title("Distribution of Image Area (Pixels)")
axes[1].set_xlabel("Total Pixels")
axes[1].set_ylabel("Frequency")
plt.show()